In [1]:
import gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
class PolicyNet(nn.Module):
    def __init__(self, action_size):
        super().__init__()
        self.l1 = nn.Linear(4, 128)
        self.l2 = nn.Linear(128, action_size)

    def forward(self, x):
        x = F.relu(self.l1(x))
        x = F.softmax(self.l2(x), dim=1)
        return x


class Agent:
    def __init__(self):
        self.gamma = 0.98
        self.lr = 0.0002
        self.action_size = 2
        # 初始化策略网络π_θ
        self.pi = PolicyNet(self.action_size)
        self.optimizer = optim.Adam(
            self.pi.parameters(),
            lr=self.lr
        )

    def get_action(self, state):
        """输入参数：环境的状态"""
        probs = self.pi(torch.tensor(state).unsqueeze(0)).squeeze(0)
        m = Categorical(probs)
        action = m.sample().item()

        return action, probs

In [3]:
env = gym.make("CartPole-v0")
state = env.reset()
agent = Agent()

action, probs = agent.get_action(state)
print("采取的动作：", "向左推" if action == 0 else "向右推")
print("采取的动作的概率：", probs[action].item())

C:\MySoftware\Anaconda3\envs\python311\Lib\site-packages\gym\envs\registration.py:593: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(
C:\MySoftware\Anaconda3\envs\python311\Lib\site-packages\gym\core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
C:\MySoftware\Anaconda3\envs\python311\Lib\site-packages\gym\wrappers\step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


采取的动作： 向左推
采取的动作的概率： 0.46288836002349854


In [4]:
G = 1.0  # G(τ)
J = -G * probs[action].log()  # −𝐺(𝜏)log𝜋𝜃(𝐴0|𝑆0)
J.backward()
agent.optimizer.step()

_, probs = agent.get_action(state)
print("采取的动作：", "向左推" if action == 0 else "向右推")
print("采取的动作的概率：", probs[action].item())

采取的动作： 向左推
采取的动作的概率： 0.46457016468048096
